In [642]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split, cross_val_score, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,confusion_matrix,classification_report
import pickle

In [643]:
df = pd.read_csv("sensor.csv")

In [644]:
print("Original shape:", df.shape)
print(df.head())

Original shape: (27150, 15)
   Unnamed: 0  session_id          run_type scenario  time_ms  sensor_id  \
0           0           1  C_front_far_deep     safe      285          1   
1           1           1  C_front_far_deep     safe      564          1   
2           2           1  C_front_far_deep     safe      839          1   
3           3           1  C_front_far_deep     safe     1114          1   
4           4           1  C_front_far_deep     safe     1408          1   

   echo_us  valid  dist_cm  dist_f_cm  baseline_cm  enter_thr_cm  exit_thr_cm  \
0     3619      1    267.8      267.8          0.0           0.0          0.0   
1     3601      1    266.5      267.5          0.0           0.0          0.0   
2     3605      1    266.8      267.3          0.0           0.0          0.0   
3        0      0      0.0      267.3          0.0           0.0          0.0   
4     3613      1    267.4      267.3          0.0           0.0          0.0   

   danger  event  
0     0.0

In [645]:
# keep only valid readings
df = df[df["valid"] == 1].copy()

In [646]:
# remove rows before baseline is learned
df = df[df["baseline_cm"] > 0].copy()

In [647]:
# remove impossible distances if any slipped in
df = df[(df["dist_f_cm"] > 20) & (df["dist_f_cm"] <= 600)].copy()

In [648]:
print("After cleaning:", df.shape)

After cleaning: (23014, 15)


In [649]:
# =========================
# 4. FEATURE ENGINEERING
# =========================

In [650]:
# main distance difference from baseline
df["delta"] = df["dist_f_cm"] - df["baseline_cm"]

In [651]:
df.head()

,Unnamed: 0,session_id,run_type,scenario,time_ms,sensor_id,echo_us,valid,dist_cm,dist_f_cm,baseline_cm,enter_thr_cm,exit_thr_cm,danger,event,delta
37,112,1,C_front_far_deep,safe,31798,1,3646,1,269.8,267.5,268.3,120.0,140.0,0.0,0,-0.8
38,113,1,C_front_far_deep,safe,32080,1,3629,1,268.5,267.8,268.3,208.3,228.3,0.0,0,-0.5
39,114,1,C_front_far_deep,safe,32363,1,3657,1,270.6,268.5,268.3,208.3,228.3,0.0,0,0.2
40,115,1,C_front_far_deep,safe,32632,1,3659,1,270.8,269.1,268.3,208.3,228.3,0.0,0,0.8
41,116,1,C_front_far_deep,safe,32902,1,3627,1,268.4,268.9,268.3,208.3,228.3,0.0,0,0.6


In [652]:
# first row contains a hardcoded enter and exit thresholds. must remove them
#df = df[df["delta"] != 0].copy()

In [653]:
#df.head()

In [654]:
# main distance difference from baseline
df["delta"] = df["dist_f_cm"] - df["baseline_cm"]

In [655]:
# normalized delta (helps when baseline changes across sessions/sensors)
df["delta_norm"] = df["delta"] / df["baseline_cm"]

In [656]:
# distance to thresholds
df["dist_to_enter"] = df["dist_f_cm"] - df["enter_thr_cm"]
df["dist_to_exit"] = df["dist_f_cm"] - df["exit_thr_cm"]

In [657]:
group_cols = []
if "session_id" in df.columns:
    group_cols.append("session_id")

In [658]:
df["velocity"] = df.groupby(group_cols)["dist_f_cm"].diff()
df["acceleration"] = df.groupby(group_cols)["velocity"].diff()

In [659]:
df.head()

,Unnamed: 0,session_id,run_type,scenario,time_ms,sensor_id,echo_us,valid,dist_cm,dist_f_cm,...,enter_thr_cm,exit_thr_cm,danger,event,delta,delta_norm,dist_to_enter,dist_to_exit,velocity,acceleration
37,112,1,C_front_far_deep,safe,31798,1,3646,1,269.8,267.5,...,120.0,140.0,0.0,0,-0.8,-0.002982,147.5,127.5,NaN,NaN
38,113,1,C_front_far_deep,safe,32080,1,3629,1,268.5,267.8,...,208.3,228.3,0.0,0,-0.5,-0.001864,59.5,39.5,0.3,NaN
39,114,1,C_front_far_deep,safe,32363,1,3657,1,270.6,268.5,...,208.3,228.3,0.0,0,0.2,0.000745,60.2,40.2,0.7,0.4
40,115,1,C_front_far_deep,safe,32632,1,3659,1,270.8,269.1,...,208.3,228.3,0.0,0,0.8,0.002982,60.8,40.8,0.6,-0.1
41,116,1,C_front_far_deep,safe,32902,1,3627,1,268.4,268.9,...,208.3,228.3,0.0,0,0.6,0.002236,60.6,40.6,-0.2,-0.8


In [660]:
print("\nAfter feature engineering:")
print(df.head())
df.shape


After feature engineering:
    Unnamed: 0  session_id          run_type scenario  time_ms  sensor_id  \
37         112           1  C_front_far_deep     safe    31798          1   
38         113           1  C_front_far_deep     safe    32080          1   
39         114           1  C_front_far_deep     safe    32363          1   
40         115           1  C_front_far_deep     safe    32632          1   
41         116           1  C_front_far_deep     safe    32902          1   

    echo_us  valid  dist_cm  dist_f_cm  ...  enter_thr_cm  exit_thr_cm  \
37     3646      1    269.8      267.5  ...         120.0        140.0   
38     3629      1    268.5      267.8  ...         208.3        228.3   
39     3657      1    270.6      268.5  ...         208.3        228.3   
40     3659      1    270.8      269.1  ...         208.3        228.3   
41     3627      1    268.4      268.9  ...         208.3        228.3   

    danger  event  delta  delta_norm  dist_to_enter  dist_to_exi

(23014, 21)

In [661]:
feature_fill_cols = ["velocity", "acceleration"]
for col in feature_fill_cols:
    if col in df.columns:
        df[col] = df[col].fillna(0)

In [662]:
print("\nAfter NaN handling:", df.shape)


After NaN handling: (23014, 21)


In [663]:
df.head()

,Unnamed: 0,session_id,run_type,scenario,time_ms,sensor_id,echo_us,valid,dist_cm,dist_f_cm,...,enter_thr_cm,exit_thr_cm,danger,event,delta,delta_norm,dist_to_enter,dist_to_exit,velocity,acceleration
37,112,1,C_front_far_deep,safe,31798,1,3646,1,269.8,267.5,...,120.0,140.0,0.0,0,-0.8,-0.002982,147.5,127.5,0.0,0.0
38,113,1,C_front_far_deep,safe,32080,1,3629,1,268.5,267.8,...,208.3,228.3,0.0,0,-0.5,-0.001864,59.5,39.5,0.3,0.0
39,114,1,C_front_far_deep,safe,32363,1,3657,1,270.6,268.5,...,208.3,228.3,0.0,0,0.2,0.000745,60.2,40.2,0.7,0.4
40,115,1,C_front_far_deep,safe,32632,1,3659,1,270.8,269.1,...,208.3,228.3,0.0,0,0.8,0.002982,60.8,40.8,0.6,-0.1
41,116,1,C_front_far_deep,safe,32902,1,3627,1,268.4,268.9,...,208.3,228.3,0.0,0,0.6,0.002236,60.6,40.6,-0.2,-0.8


In [664]:
feature_cols = [
    "dist_f_cm",
    "delta",
    "delta_norm",
    "velocity",
    "acceleration"
]


In [665]:
if "state" not in df.columns:
    df["state"] = (df["danger"] > 0).astype(int)
    label_col = "state"

In [666]:
print("\nUsing features:", feature_cols)
print("Using label:", label_col)


Using features: ['dist_f_cm', 'delta', 'delta_norm', 'velocity', 'acceleration']
Using label: state


In [667]:
df.head(1000)

,Unnamed: 0,session_id,run_type,scenario,time_ms,sensor_id,echo_us,valid,dist_cm,dist_f_cm,...,exit_thr_cm,danger,event,delta,delta_norm,dist_to_enter,dist_to_exit,velocity,acceleration,state
37,112,1,C_front_far_deep,safe,31798,1,3646,1,269.8,267.5,...,140.0,0.0,0,-0.8,-0.002982,147.5,127.5,0.0,0.0,0
38,113,1,C_front_far_deep,safe,32080,1,3629,1,268.5,267.8,...,228.3,0.0,0,-0.5,-0.001864,59.5,39.5,0.3,0.0,0
39,114,1,C_front_far_deep,safe,32363,1,3657,1,270.6,268.5,...,228.3,0.0,0,0.2,0.000745,60.2,40.2,0.7,0.4,0
40,115,1,C_front_far_deep,safe,32632,1,3659,1,270.8,269.1,...,228.3,0.0,0,0.8,0.002982,60.8,40.8,0.6,-0.1,0
41,116,1,C_front_far_deep,safe,32902,1,3627,1,268.4,268.9,...,228.3,0.0,0,0.6,0.002236,60.6,40.6,-0.2,-0.8,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1183,4708,4,C_front_far_deep,approach_danger,1325257,1,2429,1,179.7,205.3,...,225.9,0.0,0,-60.6,-0.227905,-0.6,-20.6,-8.5,-0.2,0
1184,4709,4,C_front_far_deep,approach_danger,1325539,1,2299,1,170.1,196.5,...,225.9,1.0,1,-69.4,-0.261000,-9.4,-29.4,-8.8,-0.3,1
1185,4710,4,C_front_far_deep,approach_danger,1325829,1,2357,1,174.4,191.0,...,225.9,1.0,0,-74.9,-0.281685,-14.9,-34.9,-5.5,3.3,1
1186,4711,4,C_front_far_deep,approach_danger,1326107,1,2348,1,173.8,186.7,...,225.9,1.0,0,-79.2,-0.297856,-19.2,-39.2,-4.3,1.2,1


In [668]:
# =========================
# 7. NORMALIZE FEATURES
# =========================
# z-score normalization
feature_means = df[feature_cols].mean()
feature_stds = df[feature_cols].std().replace(0, 1)

df_norm = df.copy()
df_norm[feature_cols] = (df[feature_cols] - feature_means) / feature_stds

print("\nNormalized sample:")
print(df_norm[feature_cols].head())


Normalized sample:
    dist_f_cm     delta  delta_norm  velocity  acceleration
37   1.245332  0.598460    0.599296  0.014370     -0.004162
38   1.254461  0.608338    0.608296  0.077646     -0.004162
39   1.275762  0.631386    0.629296  0.162014      0.089441
40   1.294020  0.651141    0.647296  0.140922     -0.027562
41   1.287934  0.644556    0.641296 -0.027814     -0.191367


In [669]:
df_norm.head()

,Unnamed: 0,session_id,run_type,scenario,time_ms,sensor_id,echo_us,valid,dist_cm,dist_f_cm,...,exit_thr_cm,danger,event,delta,delta_norm,dist_to_enter,dist_to_exit,velocity,acceleration,state
37,112,1,C_front_far_deep,safe,31798,1,3646,1,269.8,1.245332,...,140.0,0.0,0,0.598460,0.599296,147.5,127.5,0.014370,-0.004162,0
38,113,1,C_front_far_deep,safe,32080,1,3629,1,268.5,1.254461,...,228.3,0.0,0,0.608338,0.608296,59.5,39.5,0.077646,-0.004162,0
39,114,1,C_front_far_deep,safe,32363,1,3657,1,270.6,1.275762,...,228.3,0.0,0,0.631386,0.629296,60.2,40.2,0.162014,0.089441,0
40,115,1,C_front_far_deep,safe,32632,1,3659,1,270.8,1.294020,...,228.3,0.0,0,0.651141,0.647296,60.8,40.8,0.140922,-0.027562,0
41,116,1,C_front_far_deep,safe,32902,1,3627,1,268.4,1.287934,...,228.3,0.0,0,0.644556,0.641296,60.6,40.6,-0.027814,-0.191367,0


In [670]:
print(group_cols)

['session_id']


In [671]:
# =========================
# 8. BUILD TIME WINDOWS
# =========================
WINDOW_SIZE = 10   # 10 time steps per sample
STEP_SIZE = 1      # sliding window step

X = []
y = []

# keep windows separated by session and sensor
grouped = df_norm.groupby(group_cols)

In [672]:
for group_key, group in grouped:
    group = group.sort_values("time_ms").reset_index(drop=True)

    # need enough rows
    if len(group) < WINDOW_SIZE:
        continue

    for start in range(0, len(group) - WINDOW_SIZE + 1, STEP_SIZE):
        end = start + WINDOW_SIZE

        window_features = group.iloc[start:end][feature_cols].values
        window_label = group.iloc[end - 1][label_col]   # label from last step

        X.append(window_features)
        y.append(window_label)

In [673]:
X = np.array(X, dtype=np.float32)
y = np.array(y, dtype=np.int64)

print("\nWindowed dataset shapes:")
print("X shape:", X.shape)   # (samples, time_steps, features)
print("y shape:", y.shape)


Windowed dataset shapes:
X shape: (22204, 10, 5)
y shape: (22204,)


In [674]:
# =========================
# 9. SAVE OUTPUTS
# =========================
# cleaned flat table
df.to_csv("sensor_preprocessed_flat.csv", index=False)

# normalized flat table
df_norm.to_csv("sensor_preprocessed_normalized.csv", index=False)

# SNN-ready arrays
np.save("X_snn.npy", X)
np.save("y_snn.npy", y)

print("\nSaved files:")
print("sensor_preprocessed_flat.csv")
print("sensor_preprocessed_normalized.csv")
print("X_snn.npy")
print("y_snn.npy")


Saved files:
sensor_preprocessed_flat.csv
sensor_preprocessed_normalized.csv
X_snn.npy
y_snn.npy


In [675]:
print(X[50].shape)
print(X[50])
print("Label:", y[155])

(10, 5)
[[ 1.278805    0.63467866  0.63229614 -0.00672242  0.1830439 ]
 [ 1.2696761   0.6248009   0.62329596 -0.04890644 -0.0509632 ]
 [ 1.2392464   0.5918753   0.5932954  -0.19655049 -0.16796674]
 [ 1.2118596   0.56224227  0.5662949  -0.17545849  0.01923893]
 [ 1.1662151   0.5128538   0.52129406 -0.30201054 -0.14456603]
 [ 1.1449143   0.48980588  0.5002937  -0.13327447  0.1830439 ]
 [ 1.1296993   0.47334304  0.48529342 -0.09109046  0.04263964]
 [ 1.1418712   0.48651332  0.49729362  0.09873761  0.2064446 ]
 [ 1.1601291   0.5062687   0.51529396  0.14092162  0.04263964]
 [ 1.1510001   0.496391    0.5062938  -0.04890644 -0.21476817]]
Label: 0


In [676]:
unique, counts = np.unique(y, return_counts=True)
print(dict(zip(unique, counts)))

{np.int64(0): np.int64(18129), np.int64(1): np.int64(4075)}


In [677]:
# total number of windows
n = len(X)

# 70% train, 15% validation, 15% test
train_end = int(0.70 * n)
val_end = int(0.85 * n)

X_train = X[:train_end]
y_train = y[:train_end]

X_val = X[train_end:val_end]
y_val = y[train_end:val_end]

X_test = X[val_end:]
y_test = y[val_end:]

print("Train:", X_train.shape, y_train.shape)
print("Val  :", X_val.shape, y_val.shape)
print("Test :", X_test.shape, y_test.shape)

Train: (15542, 10, 5) (15542,)
Val  : (3331, 10, 5) (3331,)
Test : (3331, 10, 5) (3331,)


In [678]:
def label_counts(name, labels):
    unique, counts = np.unique(labels, return_counts=True)
    print(name, dict(zip(unique, counts)))

label_counts("Train", y_train)
label_counts("Val", y_val)
label_counts("Test", y_test)

Train {np.int64(0): np.int64(12707), np.int64(1): np.int64(2835)}
Val {np.int64(0): np.int64(2683), np.int64(1): np.int64(648)}
Test {np.int64(0): np.int64(2739), np.int64(1): np.int64(592)}


In [705]:
# Save original shapes
n_train, t_train, f_train = X_train.shape
n_val, t_val, f_val = X_val.shape
n_test, t_test, f_test = X_test.shape

In [680]:
# Reshape 3D -> 2D
X_train_2d = X_train.reshape(-1, f_train)
X_val_2d = X_val.reshape(-1, f_val)
X_test_2d = X_test.reshape(-1, f_test)

In [681]:
from sklearn.preprocessing import StandardScaler
# Fit scaler ONLY on training data
scaler = StandardScaler()
X_train_scaled_2d = scaler.fit_transform(X_train_2d)

In [682]:
# Apply same scaler to val/test
X_val_scaled_2d = scaler.transform(X_val_2d)
X_test_scaled_2d = scaler.transform(X_test_2d)

In [683]:
# Reshape back 2D -> 3D
X_train_scaled = X_train_scaled_2d.reshape(n_train, t_train, f_train)
X_val_scaled = X_val_scaled_2d.reshape(n_val, t_val, f_val)
X_test_scaled = X_test_scaled_2d.reshape(n_test, t_test, f_test)

print("Scaled train shape:", X_train_scaled.shape)
print("Scaled val shape  :", X_val_scaled.shape)
print("Scaled test shape :", X_test_scaled.shape)

Scaled train shape: (15542, 10, 5)
Scaled val shape  : (3331, 10, 5)
Scaled test shape : (3331, 10, 5)


In [684]:
# X_train_scaled = X_train
# X_val_scaled = X_val
# X_test_scaled = X_test

In [685]:
print("Before scaling:")
print(X_train[1])

print("\nAfter scaling:")
print(X_train_scaled[1])

Before scaling:
[[ 1.2544613   0.6083381   0.6082957   0.07764561 -0.00416178]
 [ 1.2757621   0.6313861   0.62929606  0.16201364  0.08944106]
 [ 1.2940198   0.65114146  0.6472964   0.14092162 -0.02756249]
 [ 1.287934    0.64455634  0.6412963  -0.02781443 -0.19136745]
 [ 1.3001058   0.6577266   0.65329653  0.09873761  0.13624248]
 [ 1.3153207   0.6741894   0.6682968   0.11982962  0.01923893]
 [ 1.3335785   0.69394475  0.6862971   0.14092162  0.01923893]
 [ 1.3548793   0.71699274  0.7072975   0.16201364  0.01923893]
 [ 1.388352    0.7532109   0.74029815  0.24638167  0.08944106]
 [ 1.397481    0.7630886   0.7492983   0.07764561 -0.19136745]]

After scaling:
[[ 1.2250786   0.60299844  0.6031666   0.08856785 -0.00173441]
 [ 1.2463498   0.6258446   0.62412006  0.1720492   0.08830743]
 [ 1.2645822   0.645427    0.6420802   0.15117885 -0.02424488]
 [ 1.2585047   0.63889956  0.6360935  -0.01578381 -0.1818181 ]
 [ 1.2706598   0.6519545   0.6480669   0.10943818  0.13332835]
 [ 1.2858535   0.66827

In [686]:
!pip install torch torchvision torchaudio

In [687]:
import torch

In [688]:
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.long)

X_val_t = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.long)

X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.long)

print(X_train_t.shape, y_train_t.shape)

torch.Size([15542, 10, 5]) torch.Size([15542])


In [689]:
#Create DataLoaders
from torch.utils.data import TensorDataset, DataLoader

batch_size = 64

train_dataset = TensorDataset(X_train_t, y_train_t)
val_dataset = TensorDataset(X_val_t, y_val_t)
test_dataset = TensorDataset(X_test_t, y_test_t)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [690]:
#Handle Class Imbalance
#penalize mistakes on danger more strongly

In [691]:
from sklearn.utils.class_weight import compute_class_weight

classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weights = torch.tensor(weights, dtype=torch.float32)

print(class_weights)

tensor([0.6116, 2.7411])


In [692]:
#Loss function with class weights
import torch.nn as nn

criterion = nn.CrossEntropyLoss(weight=class_weights)

In [693]:
!pip install snntorch

In [694]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import snntorch as snn
from snntorch import surrogate
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [695]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [696]:
class_weights = class_weights.to(device)

In [697]:
#the SNN model
class SimpleSNN(nn.Module):
    def __init__(self, input_size=5, hidden_size=32, output_size=2, beta=0.9):
        super().__init__()

        # Fully connected layers
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, output_size)

        # Surrogate gradient for backpropagation through spikes
        spike_grad = surrogate.fast_sigmoid()

        # LIF neurons
        self.lif1 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        self.lif2 = snn.Leaky(beta=beta, spike_grad=spike_grad)
        self.lif3 = snn.Leaky(beta=beta, spike_grad=spike_grad)

    def forward(self, x):
        batch_size = x.size(0)
        time_steps = x.size(1)

        mem1 = self.lif1.init_leaky()
        mem2 = self.lif2.init_leaky()
        mem3 = self.lif3.init_leaky()

        mem3_rec = []

        for t in range(time_steps):
            cur1 = self.fc1(x[:, t, :])
            spk1, mem1 = self.lif1(cur1, mem1)

            cur2 = self.fc2(spk1)
            spk2, mem2 = self.lif2(cur2, mem2)

            cur3 = self.fc3(spk2)
            spk3, mem3 = self.lif3(cur3, mem3)

            mem3_rec.append(mem3)

        mem3_rec = torch.stack(mem3_rec, dim=0)   # (time_steps, batch, output_size)
        return mem3_rec



In [698]:
#Create the model
model = SimpleSNN(input_size=5, hidden_size=32, output_size=2, beta=0.9).to(device)
print(model)

SimpleSNN(
  (fc1): Linear(in_features=5, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=32, bias=True)
  (fc3): Linear(in_features=32, out_features=2, bias=True)
  (lif1): Leaky()
  (lif2): Leaky()
  (lif3): Leaky()
)


In [699]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [700]:
#Training loop
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    for X_batch, y_batch in loader:
        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        mem_rec = model(X_batch)          # shape: (time_steps, batch, 2)
        output = mem_rec[-1]              # last time step -> (batch, 2)

        loss = criterion(output, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)

        preds = torch.argmax(output, dim=1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(y_batch.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)

    return epoch_loss, epoch_acc

In [701]:
# Validation loop
def evaluate(model, loader, criterion, device):
    model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            mem_rec = model(X_batch)
            output = mem_rec[-1]

            loss = criterion(output, y_batch)
            running_loss += loss.item() * X_batch.size(0)

            preds = torch.argmax(output, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)

    return epoch_loss, epoch_acc, all_labels, all_preds

In [702]:
num_epochs = 20

train_losses = []
val_losses = []
train_accs = []
val_accs = []

for epoch in range(num_epochs):
    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accs.append(train_acc)
    val_accs.append(val_acc)

    print(f"Epoch [{epoch+1}/{num_epochs}] "
          f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
          f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

Epoch [1/20] Train Loss: 0.1282 | Train Acc: 0.9611 | Val Loss: 0.0371 | Val Acc: 0.9862
Epoch [2/20] Train Loss: 0.0357 | Train Acc: 0.9853 | Val Loss: 0.0275 | Val Acc: 0.9934
Epoch [3/20] Train Loss: 0.0327 | Train Acc: 0.9858 | Val Loss: 0.0269 | Val Acc: 0.9904
Epoch [4/20] Train Loss: 0.0314 | Train Acc: 0.9869 | Val Loss: 0.0247 | Val Acc: 0.9853
Epoch [5/20] Train Loss: 0.0303 | Train Acc: 0.9864 | Val Loss: 0.0310 | Val Acc: 0.9925
Epoch [6/20] Train Loss: 0.0305 | Train Acc: 0.9867 | Val Loss: 0.0217 | Val Acc: 0.9874
Epoch [7/20] Train Loss: 0.0291 | Train Acc: 0.9866 | Val Loss: 0.0223 | Val Acc: 0.9865
Epoch [8/20] Train Loss: 0.0296 | Train Acc: 0.9860 | Val Loss: 0.0256 | Val Acc: 0.9889
Epoch [9/20] Train Loss: 0.0279 | Train Acc: 0.9869 | Val Loss: 0.0270 | Val Acc: 0.9871
Epoch [10/20] Train Loss: 0.0293 | Train Acc: 0.9866 | Val Loss: 0.0285 | Val Acc: 0.9895
Epoch [11/20] Train Loss: 0.0269 | Train Acc: 0.9869 | Val Loss: 0.0246 | Val Acc: 0.9871
Epoch [12/20] Train

In [703]:
test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader, criterion, device)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Acc : {test_acc:.4f}")

Test Loss: 0.0362
Test Acc : 0.9895


In [704]:
print(feature_cols)

['dist_f_cm', 'delta', 'delta_norm', 'velocity', 'acceleration']
